In [9]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import torch

from src.models.baseline_cnn import BaselineCNN

from src.training.losses import build_loss
from src.training.optimizers import build_optimizer
from src.training.train import train_one_epoch

from src.utils.config import TRAIN_IMAGES, TRAIN_LABELS

from src.data.labels import load_labels
from src.data.split_data import split_patients
from src.data.dataset import RSNADataset
from src.data.dataloader import build_dataloader

In [10]:
labels, _ = load_labels(TRAIN_LABELS)

train_ids, val_ids, test_ids = split_patients(labels)

In [11]:
train_dataset = RSNADataset(
    patient_ids=train_ids,
    labels=labels,
    image_dir=TRAIN_IMAGES
)

train_loader = build_dataloader(
    dataset=train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=4
)

In [12]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [13]:
model = BaselineCNN().to(device)

criterion = build_loss("bce")

optimizer = build_optimizer(
    model,
    optimizer_name="adam",
    learning_rate=1e-3
)

In [14]:
print(f"Training samples : {len(train_dataset)}")
print(f"Number of batches : {len(train_loader)}")
print(f"Device : {device}")

Training samples : 18678
Number of batches : 1168
Device : cuda


In [15]:
model = BaselineCNN()

print(model)

BaselineCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): AdaptiveAvgPool2d(output_size=(1, 1))
    (1): Flatten(start_dim=1, end_dim=-1)
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=1, bias=True)
    (5): Sigmoid()
  )
)


In [16]:
model = BaselineCNN().to(device)

loss = train_one_epoch(
    model=model,
    dataloader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device
)

print(loss)

Batch [100/1168] Loss: 0.6956
Batch [200/1168] Loss: 0.7004
Batch [300/1168] Loss: 0.7005
Batch [400/1168] Loss: 0.7031
Batch [500/1168] Loss: 0.7052
Batch [600/1168] Loss: 0.6981
Batch [700/1168] Loss: 0.7028
Batch [800/1168] Loss: 0.7053
Batch [900/1168] Loss: 0.7029
Batch [1000/1168] Loss: 0.6981
Batch [1100/1168] Loss: 0.6957
0.703860068208959


In [17]:
import torch

print(torch.__version__)

print(torch.cuda.is_available())

print(torch.cuda.device_count())

print(torch.cuda.get_device_name(0))

print(torch.version.cuda)

2.13.0+cu126
True
1
NVIDIA GeForce RTX 4050 Laptop GPU
12.6


In [18]:
print(train_loader)
print("batch_size:", train_loader.batch_size, type(train_loader.batch_size))
print("num_workers:", train_loader.num_workers, type(train_loader.num_workers))
print("dataset:", type(train_loader.dataset))
print("dataset length:", len(train_loader.dataset))

batch_size: 16 <class 'int'>
num_workers: 4 <class 'int'>
dataset: <class 'src.data.dataset.RSNADataset'>
dataset length: 18678
